# **Modelling and Evaluation for K-Means Clustering Task**

## Objectives

* Construct a model that can predict distinct groups/clusters from smartwatch health data for targetted advertisements

## Inputs

* unclean_smartwatch_health_data.csv

## Outputs

* K-Means Clustering Model PKL

---

# Change working directory

We need to change the working directory from its current folder to its parent folder.

In [ ]:
import os
current_dir = os.getcwd()
current_dir

We want to make the parent of the current directory the new current directory.

In [ ]:
os.chdir(os.path.dirname(current_dir))
print("You set a new current directory")

Confirm the new current directory.

In [ ]:
current_dir = os.getcwd()
current_dir

Setup needed variables.

In [ ]:
InputFolder = "inputs/"
OutputFolder = "outputs/"
UntouchedData = InputFolder + "smartwatch_health_data_untouched/"
CleanedData = InputFolder + "cleaned_data/"

In [ ]:
version = "v1"

OutputFolder = f"outputs/{version}/cluster_task/"
if not os.path.exists(OutputFolder):
    os.makedirs(OutputFolder)

## Step 1: Load Data

Load data and drop User ID

In [ ]:
import pandas as pd

df = pd.read_csv(UntouchedData + "unclean_smartwatch_health_data.csv").drop(
    "User ID", axis=1)
df.head()

## Step 2: Data Cleaning and Engineering

Declare custom transformers

In [ ]:
import numpy as np
from sklearn.base import BaseEstimator, TransformerMixin

# Replace 'Very High' with 10 in 'Stress Level' and convert columns to numeric
class DataTypeTransformer(BaseEstimator, TransformerMixin):
    def __init__(self):
        pass

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        X = X.copy()
        # Replace 'Very High' with 10 in 'Stress Level'
        X['Stress Level'] = X['Stress Level'].replace('Very High', 10)

        # Convert columns to numeric, handling non-numeric values
        X['Sleep Duration (hours)'] = pd.to_numeric(
            X['Sleep Duration (hours)'], errors='coerce')
        X['Stress Level'] = pd.to_numeric(X['Stress Level'], errors='coerce')

        return X


# Impute missing values (not for Activity Level)
from sklearn.impute import KNNImputer

class KNNImputerTransformer(BaseEstimator, TransformerMixin):
    def __init__(self, n_neighbors=3):
        self.n_neighbors = n_neighbors
        self.imputer = KNNImputer(n_neighbors=self.n_neighbors)

    def fit(self, X, y=None):
        # Identify numerical columns
        self.num_cols = X.select_dtypes(include=[np.number]).columns
        self.imputer.fit(X[self.num_cols])
        return self

    def transform(self, X):
        X = X.copy()
        # Impute numerical columns
        X[self.num_cols] = self.imputer.transform(X[self.num_cols])
        return X


# Handle Outliers
from feature_engine.outliers import Winsorizer

class WinsorizerTransformer(BaseEstimator, TransformerMixin):
    def __init__(self):
        self.winsorizers = {}

    def fit(self, X, y=None):
        X = X.copy()
        # Define and fit Winsorizers for each variable
        self.winsorizers['Sleep Duration (hours)'] = Winsorizer(
            capping_method='iqr', tail='both', fold=1.5,
            variables=['Sleep Duration (hours)']
        ).fit(X)
        self.winsorizers['Heart Rate (BPM)'] = Winsorizer(
            capping_method='iqr', tail='right', fold=1.5,
            variables=['Heart Rate (BPM)']
        ).fit(X)
        self.winsorizers['Blood Oxygen Level (%)'] = Winsorizer(
            capping_method='iqr', tail='left', fold=1.5,
            variables=['Blood Oxygen Level (%)']
        ).fit(X)
        self.winsorizers['Step Count'] = Winsorizer(
            capping_method='iqr', tail='right', fold=1.5,
            variables=['Step Count']
        ).fit(X)
        return self

    def transform(self, X):
        X = X.copy()
        # Apply Winsorizers
        for winsorizer in self.winsorizers.values():
            X = winsorizer.transform(X)
        return X


class FloatToInt(BaseEstimator, TransformerMixin):
    def __init__(self, columns=['Stress Level']):
        self.columns = columns

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        X = X.copy()
        # Convert specified columns to integer
        for col in self.columns:
            X[col] = X[col].astype(int)
        return X

# Predict missing values in 'Activity Level'
from sklearn.ensemble import RandomForestClassifier

class CategoricalImputer(BaseEstimator, TransformerMixin):
    def __init__(self, target_column='Activity Level', random_state=42):
        self.target_column = target_column
        self.random_state = random_state
        self.classifier = RandomForestClassifier(random_state=self.random_state)
        self.original_categories = None

    def fit(self, X, y=None):
        X = X.copy()
        # Save original categories
        X[self.target_column] = X[self.target_column].astype('category')
        self.original_categories = X[self.target_column].cat.categories

        # Encode target column
        X[self.target_column] = X[
            self.target_column].cat.codes.replace(-1, np.nan)

        # Split data
        self.train_data = X.dropna(subset=[self.target_column])
        self.test_data = X[X[self.target_column].isna()]

        # Features and target
        self.X_train = self.train_data.drop(columns=[self.target_column])
        self.y_train = self.train_data[self.target_column].astype(int)

        # Fit classifier
        self.classifier.fit(self.X_train, self.y_train)
        return self

    def transform(self, X):
        X = X.copy()
        # Encode target column
        X[self.target_column] = X[self.target_column].astype('category')
        X[self.target_column] = X[
            self.target_column].cat.codes.replace(-1, np.nan)

        # Identify missing values
        missing_mask = X[self.target_column].isna()
        if missing_mask.any():
            X_missing = X[missing_mask]
            X_missing_features = X_missing.drop(columns=[self.target_column])
            # Predict missing values
            X.loc[missing_mask, self.target_column] = self.classifier.predict(
                X_missing_features)

        # Convert to int and decode categories
        X[self.target_column] = X[self.target_column].astype(int)
        X[self.target_column] = pd.Categorical.from_codes(
            X[self.target_column], categories=self.original_categories
        )
        return X


# Correct mis-spelled values in Activity Level
class CategoryCorrector(BaseEstimator, TransformerMixin):
    def __init__(self, column='Activity Level'):
        self.column = column
        self.class_mapping = {
            'Seddentary': 'Sedentary',
            'Highly_Active': 'Highly Active',
            'Actve': 'Active'
        }

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        X = X.copy()
        X[self.column] = X[self.column].replace(self.class_mapping)
        return X


# Smooth data using K-Nearest Neighbors
from sklearn.neighbors import NearestNeighbors

class DataSmoother(BaseEstimator, TransformerMixin):
    def __init__(self, k=3):
        self.k = k
        self.nn = NearestNeighbors(n_neighbors=self.k)

    def fit(self, X, y=None):
        X = X.copy()
        self.num_cols = X.select_dtypes(include=[np.number]).columns
        self.nn.fit(X[self.num_cols])
        return self

    def transform(self, X):
        X = X.copy()
        distances, indices = self.nn.kneighbors(X[self.num_cols])

        # Smooth numerical columns
        for i, col in enumerate(self.num_cols):
            X[col] = [np.mean(
                X.iloc[indices[row_idx]][col]) for row_idx in range(len(X))]
        return X


# Trim outliers again after smoothing
from feature_engine.outliers import OutlierTrimmer

class OutlierTrimmerTransformer(BaseEstimator, TransformerMixin):
    def __init__(self):
        self.trimmers = {}

    def fit(self, X, y=None):
        X = X.copy()
        # Define and fit Outlier Trimmers for each variable
        self.trimmers['Heart Rate (BPM)'] = OutlierTrimmer(
            capping_method='quantiles', tail='right', fold=0.05,
            variables=['Heart Rate (BPM)']
        ).fit(X)
        self.trimmers['Blood Oxygen Level (%)'] = OutlierTrimmer(
            capping_method='quantiles', tail='left', fold=0.05,
            variables=['Blood Oxygen Level (%)']
        ).fit(X)
        self.trimmers['Sleep Duration (hours)'] = OutlierTrimmer(
            capping_method='quantiles', tail='both', fold=0.05,
            variables=['Sleep Duration (hours)']
        ).fit(X)
        self.trimmers['Step Count'] = OutlierTrimmer(
            capping_method='quantiles', tail='right', fold=0.05,
            variables=['Step Count']
        ).fit(X)
        return self

    def transform(self, X):
        X = X.copy()
        # Apply Outlier Trimmers sequentially
        for trimmer in self.trimmers.values():
            X = trimmer.transform(X)
        return X

# This is important else even the categoric column will be minmax
# scaled resulting in much different components when evaluated with a PCA
from sklearn.preprocessing import MinMaxScaler , StandardScaler , RobustScaler
class DataFrameScaler(BaseEstimator, TransformerMixin):
    def __init__(self, exclude_columns=None):
        self.exclude_columns = exclude_columns
        self.scaler = RobustScaler()
    
    def fit(self, X, y=None):
        X_to_scale = X.drop(columns=self.exclude_columns)
        self.scaler.fit(X_to_scale)
        return self
    
    def transform(self, X):
        X_to_scale = X.drop(columns=self.exclude_columns)
        X_excluded = X[self.exclude_columns]
        X_scaled = pd.DataFrame(self.scaler.transform(X_to_scale),
                                columns=X_to_scale.columns)
        X_final = pd.concat([X_scaled, X_excluded.reset_index(drop=True)],
                            axis=1)
        return X_final

We researched for the optimal values for n_components and n_clusters in the engineering notebook. After carefull evaluation of both the smoothed and not-smoothed datasets, I will be using the smoothed dataset for this modelling. From viewing different QQ plots and such, I have learned the smoothed set doesnt need a numerical transformation, as it doesnt have a positive effect.

In [ ]:
from sklearn.pipeline import Pipeline
# encoding
from feature_engine.encoding import OrdinalEncoder
# pca
from sklearn.decomposition import PCA
# kmeans
from sklearn.cluster import KMeans
# scale

def CleanEnginPipeline():
    cleaning_engineering_pipeline = Pipeline([
        ('data_type_transformer', DataTypeTransformer()),
        ('knn_imputer', KNNImputerTransformer(n_neighbors=3)),
        ('winsorizer_transformer', WinsorizerTransformer()),
        ('categorical_imputer', CategoricalImputer()),
        ('category_corrector', CategoryCorrector()),
        ('data_smoother', DataSmoother(k=3)),
        ('outlier_trimmer_transformer', OutlierTrimmerTransformer()),
        ('encoder', OrdinalEncoder(encoding_method='arbitrary',
                                   variables=["Activity Level"])),
        # I have chosen OrdinalEncoder as the
        # categorical variables have an ordinal relationship
        ('minmax', DataFrameScaler(exclude_columns=["Activity Level",
                                                    "Stress Level"])),
        ('float_to_int', FloatToInt()),
    ])
    return cleaning_engineering_pipeline

I will use this cleaned data variable later.

In [ ]:
df_cleaned = CleanEnginPipeline().fit_transform(df)
df_cleaned = pd.DataFrame(df_cleaned, columns=df.columns)
df_cleaned.head()

In [ ]:
def FullPipeline():
    cleaning_engineering_pipeline = CleanEnginPipeline()
    clustering_pipeline = Pipeline([
        ('cleaning_engineering_pipeline', cleaning_engineering_pipeline),
        ('pca', PCA(n_components=3, random_state=42)),
        ('model', KMeans(n_clusters=6, random_state=42))
    ])
    return clustering_pipeline

Lets double check the components to make sure the custom transformers are doing what they are supposed to. And check the elbow and silhouette plots again, to make sure they are similar to the plot tests done in the engineering notebook.

In [ ]:
full_pipeline = FullPipeline()
pipeline_pca = Pipeline(full_pipeline.steps[:-2])
df_pca = pipeline_pca.fit_transform(df)

print(df_pca.shape,'\n', type(df_pca))

When attempting this PCA without scaling, component 0 (Activity Level, always takes 99% of the variance).

In [ ]:
n_components = 3

pca = PCA(n_components=n_components).fit(df_pca) 
x_PCA = pca.transform(df_pca) 

ComponentsList = ["Component " + str(number) for number in range(n_components)]
dfExplVarRatio = pd.DataFrame(
    data= np.round(100 * pca.explained_variance_ratio_ ,3),
    index=ComponentsList,
    columns=['Explained Variance Ratio (%)'])

PercentageOfDataExplained = dfExplVarRatio['Explained Variance Ratio (%)'].sum()

print(f"* The {n_components} components explain"
      f" {round(PercentageOfDataExplained,2)}% of the data \n")
print(dfExplVarRatio)

I can explain 100% of the data with 6 components. Component 0 is the most powerfull, with a ratio of 60.469% of the data variance. Component 1 also holds a small significant amount, I think the main reason is two variables we excluded from the scaling, as they are already in a simple interger format. I would like to try just 3 componenets to see the outcome. As I feel the other components with a low varience wont split well into distinct clusters. I can reduce feature space if three componenets is overkill.

In [ ]:
pipeline_cluster = FullPipeline()
df_analysis = pipeline_cluster.fit_transform(df)

print(df_analysis.shape,'\n', type(df_analysis))

In [ ]:
import matplotlib.pyplot as plt
from matplotlib import rcParams
import seaborn as sns
rcParams['font.family'] = ['DejaVu Sans']

plt.figure(figsize=(9, 6))
sns.lineplot(data=dfExplVarRatio,  marker="o")
plt.xticks(rotation=90)
plt.yticks(np.arange(0, 110, 10))
plt.savefig(OutputFolder + "ExplainedVarianceRatio.png")
plt.show()

In [ ]:
from yellowbrick.cluster import SilhouetteVisualizer
from yellowbrick.cluster import KElbowVisualizer
from matplotlib import rcParams
rcParams['font.family'] = ['DejaVu Sans']

print("=== Average Silhouette Score for different number of clusters ===")
visualizer = KElbowVisualizer(KMeans(random_state=42, n_init=10), k=(2,7),
                              metric='silhouette')
visualizer.fit(df_analysis) 
visualizer.show() 
plt.savefig(os.path.join(OutputFolder, "elbow_plot.png"))
plt.close()
print("\n")

for n_clusters in np.arange(start=2,stop=9):
  
  print(f"=== Silhouette plot for {n_clusters} Clusters ===")
  visualizer = SilhouetteVisualizer(estimator = KMeans(n_clusters=n_clusters,
                                    random_state=42, n_init=10),
                                    colors = 'yellowbrick')
  visualizer.fit(df_analysis)
  visualizer.show()
  plt.savefig(os.path.join(OutputFolder,
                           f"silhouette_plot_{n_clusters}_clusters.png"))
  plt.close()
  print("\n")

Looks good. The best is 7 and 8 clusters. Though i would like to try 6 first. The silhouette score advises just 2 clusters, but 4 clusters isnt that far off. Noticeable improvements return after 6 clusters.

## Train

After double checking the components and silhouettes are similar after the custom transformers. I am happy to start training the model.

I will fit the pipeline and grab the predicted cluster labels.

In [ ]:
pipeline_cluster = FullPipeline()
X = pipeline_cluster.fit_transform(df.copy())
X = pd.DataFrame(X)

# cluster labels
X['Clusters'] = pipeline_cluster['model'].labels_
print(X.shape)
X.head(3)


Checkout the cluster frequencies.

In [ ]:
import matplotlib.pyplot as plt

print(f"* Clusters frequencies \n{ X['Clusters'].value_counts(
    normalize=True).to_frame().round(2)} \n\n")
X['Clusters'].value_counts().sort_values().plot(kind='bar')
plt.savefig(os.path.join(OutputFolder, "cluster_frequencies.png"))
plt.show()

Great these 6 cluster profiles look good enough with respect to balance. It could be better, but I think this will be a good start.

Lets visualise this better by plotting a scatter plot coloured by clusters.

In [ ]:
sns.scatterplot(x=X.columns[0], y=X.columns[1],
                hue='Clusters', palette='Set1', alpha=0.6, data=X)
plt.scatter(x=pipeline_cluster.named_steps['model'].cluster_centers_[:, 0], 
            y=pipeline_cluster.named_steps['model'].cluster_centers_[:, 1],
            marker="x", s=169, linewidths=3, color="black")
plt.xlabel("PCA Component 0")
plt.ylabel("PCA Component 1")
plt.title("PCA Components colored by Clusters")
plt.savefig(os.path.join(OutputFolder, "clustered_pca_plot.png"))
plt.show()

This plot looks alright. You can make out the different colours, they could be more far apart and distinct though.

## Fit classifier

Now I will fit the classifier.

In [ ]:
df_clf = X.copy()
print(df_clf.shape)
df_clf.head(3)

Save the cluster predictions for later.

In [ ]:
all_cluster_predictions = X["Clusters"].values
all_cluster_predictions

Split the dataset but with the target variable as "Clusters".

In [ ]:

from sklearn.model_selection import train_test_split
X_train, X_test,y_train, y_test = train_test_split(
                                    df_clf.drop(['Clusters'],axis=1),
                                    df_clf['Clusters'],
                                    test_size=0.2,
                                    random_state=42
                                    )

print(X_train.shape, y_train.shape, X_test.shape, y_test.shape)

Assemble classifier pipeline. After a thorougher test of different classifiers with GridSearchCV, GradientBoosterClassifier comes out on top.

In [ ]:
from sklearn.feature_selection import SelectFromModel
from sklearn.ensemble import GradientBoostingClassifier 

def PipelineClassify():
  pipeline_base = Pipeline([
      ("feat_selection", SelectFromModel(GradientBoostingClassifier(random_state=42)) ), 
      ("model",  GradientBoostingClassifier(random_state=42) ), 
  ])
  return pipeline_base

Fit the classifier.

In [ ]:
pipeline_clf_cluster = PipelineClassify()
pipeline_clf_cluster.fit(X_train, y_train)

## Evaluate the Model

In [ ]:
from sklearn.metrics import classification_report
print(classification_report(y_train, pipeline_clf_cluster.predict(X_train)))

In [ ]:
print(classification_report(y_test, pipeline_clf_cluster.predict(X_test)))

Good generalizing capability. lets now asses the most important features that define the clusters.

In [ ]:
# Get feature importances from the classifier
importances = pipeline_clf_cluster.named_steps['model'].feature_importances_

# Get the indices of the most important features
indices = np.argsort(importances)[::-1]

# Get the names of the most important features
best_features = [X_train.columns[i] for i in indices]

print("Best features in order of importance:")
print(best_features)

These are the best features, though we dont know their original names yet, lets find that out.

First, save the best features for later.

In [ ]:
best_features_pipeline_all_variables = best_features
best_features_pipeline_all_variables

Include analysis functions.

In [ ]:
def DescriptionAllClusters(df, decimal_points=3):

    DescriptionAllClusters = pd.DataFrame(
        columns=df.drop(['Clusters'], axis=1).columns)
    # iterate on each cluster , calls Clusters_IndividualDescription()
    for cluster in df.sort_values(by='Clusters')['Clusters'].unique():

        EDA_ClusterSubset = df.query(
            f"Clusters == {cluster}").drop(['Clusters'], axis=1)
        ClusterDescription = Clusters_IndividualDescription(
            EDA_ClusterSubset, cluster, decimal_points)
        DescriptionAllClusters = pd.concat(
            [ClusterDescription, DescriptionAllClusters])

    DescriptionAllClusters.set_index(['Cluster'], inplace=True)
    return DescriptionAllClusters


def Clusters_IndividualDescription(EDA_Cluster, cluster, decimal_points):

    ClustersDescription = pd.DataFrame(columns=EDA_Cluster.columns)
    # for a given cluster, iterate over all columns
    # if the variable is numerical, calculate the IQR: display as Q1 -- Q3.
    # That will show the range for the most
    # common values for the numerical variable
    # if the variable is categorical, count the
    # frequencies and displays the top 3 most frequent
    # That will show the most common levels for the category

    for col in EDA_Cluster.columns:

        try:

            if EDA_Cluster[col].dtypes == 'object':

                top_frequencies = EDA_Cluster.dropna(
                    subset=[col])[[col]].value_counts(
                        normalize=True).nlargest(n=3)
                Description = ''

                for x in range(len(top_frequencies)):
                    freq = top_frequencies.iloc[x]
                    category = top_frequencies.index[x][0]
                    CategoryPercentage = int(round(freq*100, 0))
                    statement = f"'{category}': {CategoryPercentage}% , "
                    Description = Description + statement

                ClustersDescription.at[0, col] = Description[:-2]

            elif EDA_Cluster[col].dtypes in ['float', 'int']:
                DescStats = EDA_Cluster.dropna(subset=[col])[[col]].describe()
                Q1 = round(DescStats.iloc[4, 0], decimal_points)
                Q3 = round(DescStats.iloc[6, 0], decimal_points)
                Description = f"{Q1} -- {Q3}"
                ClustersDescription.at[0, col] = Description

        except Exception as e:
            ClustersDescription.at[0, col] = 'Not available'
            print(
                f"** Error Exception: {e} - cluster {cluster}, variable {col}")

    ClustersDescription['Cluster'] = str(cluster)

    return ClustersDescription

In [ ]:
import plotly.express as px


def cluster_distribution_per_variable(df, target):
    """
    The data should have 2 variables, the cluster predictions and
    the variable you want to analyze with, in this case we call "target".
    We use plotly express to create 2 plots:
    Cluster distribution across the target.
    Relative presence of the target level in each cluster.
    """
    df_bar_plot = df.groupby(
        ['Clusters', target]).size().reset_index(name='Count')
    df_bar_plot.columns = ['Clusters', target, 'Count']
    df_bar_plot[target] = df_bar_plot[target].astype('object')

    print(f"Clusters distribution across {target} levels")
    fig = px.bar(df_bar_plot, x='Clusters', y='Count',
                 color=target, width=800, height=500)
    fig.update_layout(xaxis=dict(tickmode='array',
                      tickvals=df['Clusters'].unique()))
    fig.show(renderer='jupyterlab')

    df_relative = (df
                   .groupby(["Clusters", target])
                   .size()
                   .unstack(fill_value=0)
                   .apply(lambda x: 100 * x / x.sum(), axis=1)
                   .stack()
                   .reset_index(name='Relative Percentage (%)')
                   .sort_values(by=['Clusters', target])
                   )

    print(f"Relative Percentage (%) of {target} in each cluster")
    fig = px.line(df_relative, x='Clusters', y='Relative Percentage (%)',
                  color=target, width=800, height=500)
    fig.update_layout(xaxis=dict(tickmode='array',
                      tickvals=df['Clusters'].unique()))
    fig.update_traces(mode='markers+lines')
    fig.show(renderer='jupyterlab')

Using the assessment functions below, I can work out what features had the most impact.

In [ ]:
df_cluster_profile = df_clf.copy()
df_cluster_profile = df_cluster_profile.filter(items=best_features + ['Clusters'], axis=1)
df_active = df_cleaned["Activity Level"].astype('object')
df_stress = df_cleaned["Stress Level"].astype('object')
df_step = df_cleaned["Step Count"].astype('object')
df_blood = df_cleaned["Blood Oxygen Level (%)"].astype('object')
df_heart = df_cleaned["Heart Rate (BPM)"].astype('object')
df_sleep = df_cleaned["Sleep Duration (hours)"].astype('object')

Activity Level

In [ ]:
pd.set_option('display.max_colwidth', None)
clusters_profile = DescriptionAllClusters(df=pd.concat([df_cluster_profile,df_active], axis=1), decimal_points=0)
clusters_profile

Stress Level

In [ ]:
pd.set_option('display.max_colwidth', None)
clusters_profile = DescriptionAllClusters(df=pd.concat([df_cluster_profile,df_stress], axis=1), decimal_points=0)
clusters_profile

Blood Oxygen level

In [ ]:
pd.set_option('display.max_colwidth', None)
clusters_profile = DescriptionAllClusters(df=pd.concat([df_cluster_profile,df_blood], axis=1), decimal_points=0)
clusters_profile

Step Count

In [ ]:
pd.set_option('display.max_colwidth', None)
clusters_profile = DescriptionAllClusters(df=pd.concat([df_cluster_profile,df_step], axis=1), decimal_points=0)
clusters_profile

Heart Rate

In [ ]:
pd.set_option('display.max_colwidth', None)
clusters_profile = DescriptionAllClusters(df=pd.concat([df_cluster_profile,df_heart], axis=1), decimal_points=0)
clusters_profile

Sleep Duration

In [ ]:
pd.set_option('display.max_colwidth', None)
clusters_profile = DescriptionAllClusters(df=pd.concat([df_cluster_profile,df_sleep], axis=1), decimal_points=0)
clusters_profile

The rest of the columns apart from Activity Level, Stress Level and a tiny bit of Bloody Oxygen Level are not used to make predictions on clusters at all, as I set the components to three as this contained enough varience.

Now to check how Stress Level and Activity Level have translated over to clusters by viewing how much of easy type of Activity and Stress Levels are within each defined cluster.

In [ ]:
df_cluster_vs=  df_cleaned.copy()
df_cluster_vs['Clusters'] = X['Clusters']
cluster_distribution_per_variable(df=df_cluster_vs, target='Stress Level')

Lets check the same with Activity Level, where I am predicting a good profile with clear seperation and distinct groups.

In [ ]:
df_cluster_vs=  df_cleaned.copy()
df_cluster_vs['Clusters'] = X['Clusters']
cluster_distribution_per_variable(df=df_cluster_vs, target='Activity Level')

In [ ]:
print(classification_report(y_test, pipeline_clf_cluster.predict(X_test)))

Great these have been clustered well, and with only needing three components, I feel this is a good result and the company can find quite a few different group types to advertise to. I will outline these in the clustering streamlit dashboard.

In [ ]:
from sklearn.metrics import confusion_matrix
# Confusion matrix
cluster_predictions_with_best_features = df_clf['Clusters']
print(confusion_matrix(cluster_predictions_with_best_features,
                       cluster_predictions_with_best_features))

In [ ]:
# Create and fit the pipeline
pipeline_clf_cluster = PipelineClassify()
pipeline_clf_cluster.fit(X_train, y_train)

# Retrieve the feature selector
feat_selector = pipeline_clf_cluster.named_steps['feat_selection']

# Get the boolean mask of selected features
support_mask = feat_selector.get_support()

# Map the selected features to their original column names
selected_features = X_train.columns[support_mask]

# Display the selected features
print("Selected Features:", selected_features.to_list())

In [ ]:
# fit the pipeline to ensure the feature selector is fitted
pipeline_clf_cluster.fit(X_train, y_train)

# transform the data using the pipeline
transformed_X_train = pipeline_clf_cluster.named_steps[
    'feat_selection'].transform(X_train)

# get the feature names from the transformed data
columns_after_data_cleaning_feat_eng = X_train.columns[
    pipeline_clf_cluster.named_steps['feat_selection'].get_support()]

# create DataFrame to display feature importance
df_feature_importance = (pd.DataFrame(data={
    'Feature': columns_after_data_cleaning_feat_eng,
    'Importance': pipeline_clf_cluster.named_steps[
                                    'model'].feature_importances_})
    .sort_values(by='Importance', ascending=False)
)

# reassign best features in importance order
best_features = df_feature_importance['Feature'].to_list()

# Most important features statement and plot
print(f"* These are the {len(best_features)} most"
      f"important features in descending order. The model was"
      f" trained on them: \n{best_features} \n")
df_feature_importance.plot(kind='bar', x='Feature', y='Importance')
plt.show()

It looks like

So, it seems you can group people with 100% accuracy into 6 different segregations of stress level and activity levels and target them for specific advertising, tailored to there active lifestyle and stress. The other features didnt prove to be helpful for this task.

---

## Final model

I will now put together the final model.

Build the full pipeline -

In [ ]:
def PipelineFinal():
  pipeline_base = Pipeline([
    ('data_type_transformer', DataTypeTransformer()),
    ('knn_imputer', KNNImputerTransformer(n_neighbors=3)),
    ('winsorizer_transformer', WinsorizerTransformer()),
    ('categorical_imputer', CategoricalImputer()),
    ('category_corrector', CategoryCorrector()),
    ('data_smoother', DataSmoother(k=3)),
    ('outlier_trimmer_transformer', OutlierTrimmerTransformer()),
    ('encoder', OrdinalEncoder(encoding_method='arbitrary',
                               variables=["Activity Level"])),
    ('minmax', DataFrameScaler(exclude_columns=["Activity Level",
                                                "Stress Level"])),
    ('float_to_int', FloatToInt()),
    ("feat_selection", SelectFromModel(
      GradientBoostingClassifier(random_state=42)) ), 
    ("model",  GradientBoostingClassifier(random_state=42) ), 
  ])
  return pipeline_base

Drop not needed features, learnt from the PCA method.

In [ ]:
# Fit the final pipeline
pipeline_cluster = PipelineFinal()
pipeline_first = Pipeline(pipeline_cluster.steps[:10])
df_final = pipeline_first.fit_transform(df.copy())
X = pd.DataFrame(df_final)
X.drop(["Sleep Duration (hours)", "Heart Rate (BPM)"], axis=1, inplace=True)

Something strange is the fact that removing the features with low varience like the PCA suggested and leaving it with the two most powerful ones (Stress Level and Activity Level) actually worsens the model. It performs best with just Sleep Duration and Heart Rate being dropped.

Fit the K-Means model and collect cluster labels.

In [ ]:
fit_kmeans = KMeans(n_clusters=6, random_state=42, n_init=10)
fit_kmeans.fit(X)
X['Clusters'] = fit_kmeans.labels_

Split the data -

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X.drop(['Clusters'], axis=1),
    X['Clusters'],
    test_size=0.2,
    random_state=42
)

Perform the last two steps of the pipeline.

In [ ]:
pipeline2_cluster = PipelineFinal()
pipeline_final = Pipeline(pipeline_cluster.steps[-2:])
pipeline_final.fit(X_train, y_train)

Evaluate the model.

In [ ]:
y_pred = pipeline_final.predict(X_test)
confusion_matrix(y_test, y_pred)

In [ ]:
print(classification_report(y_test, y_pred))

Fit the pipeline again to ensure feature selection.

In [ ]:
pipeline_final.fit(X_train, y_train)

Transform the data using the best features

In [ ]:
transformed_X_train = pipeline_final.named_steps['feat_selection'].transform(X_train)

Ensure original column names are aligned with the transformed data

In [ ]:
columns_after_data_cleaning_feat_eng = X_train.columns[pipeline_final.named_steps['feat_selection'].get_support()]

Assess -

In [ ]:
# Get feature importances from the model
df_feature_importance = pd.DataFrame({
    'Feature': columns_after_data_cleaning_feat_eng,
    'Importance': pipeline_final.named_steps['model'].feature_importances_
}).sort_values(by='Importance', ascending=False)

# Ensure that "Feature" column contains original column names
df_feature_importance.reset_index(drop=True, inplace=True)

# Reassign best features in importance order
best_features = df_feature_importance['Feature'].to_list()

# Most important features statement and plot
print(f"* These are the {len(best_features)} most important features in descending order. "
      f"The model was trained on them: \n{best_features} \n")

# Plot the bar chart
ax = df_feature_importance.plot(kind='bar', x='Feature', y='Importance', legend=False)
plt.xlabel("Features")
plt.ylabel("Importance")

# Annotate each bar with its corresponding label
for idx, row in df_feature_importance.iterrows():
    ax.text(idx, row['Importance'] + 0.01, row['Feature'], ha='center', va='bottom')

In [ ]:
print(classification_report(y_test, pipeline_final.predict(X_test)))

Great! let's wrap this up.

In [ ]:
import joblib

joblib.dump(pipeline_final, OutputFolder + "final_cluster_model.pkl")

In [ ]:
X.to_csv(OutputFolder + "final_cluster_data.csv", index=False)
df_cluster_profile.to_csv(OutputFolder + "df_cluster_profile.csv", index=False)
X["Clusters"].to_csv(OutputFolder + "final_cluster_series.csv", index=False)

---

This ML task I felt was performed well, producing a decent model with high precision for grouping individuals by their smart watch data to be targeted for specific advertisements tailored to them.

---